# TP 12 — RNN (réseau récurrent simple)
**Deep Learning · Séquences**


| Exemple | Diapo | Problème |
|---|---|---|
| Exemple 1 | 84 | Régression : ventes de 3 jours (1, 2, 3 centaines de TND) → prédire le jour suivant (réel : 4) |
| Exemple 2 | 85 | Classification : la série 1, 2, 3 est-elle croissante ? |

**Objectifs** : programmer **en Python pur** la lecture de la séquence par la cellule, puis l’entraînement de la couche de sortie (3 epochs), sur les exemples des diapos ; refaire le même entraînement avec **Keras** et comparer les deux implémentations.

## Rappel de cours

Un RNN lit une séquence **pas à pas** et garde une mémoire $h_t$ qui résume ce qu’il a déjà lu :
$$h_t = \tanh(W_x\,x_t + W_h\,h_{t-1} + b)$$
Les **mêmes poids** $W_x, W_h, b$ servent à chaque pas de temps (poids partagés). La dernière mémoire $h_3$ alimente une **couche de sortie** :
- régression : $\hat y = v\cdot h + c$, perte $(y - \hat y)^2$
- classification : $p = \sigma(v\cdot h + c)$, perte $-\ln p$ (quand $y = 1$)

**Simplification des diapos** (pour calculer à la main) : $W_x = 1$, $W_h = 0{,}5$, $b = 0$, **sans tanh**, donc
$$h_t = 0{,}5\,h_{t-1} + x_t$$
La cellule est **fixée** ; on n’entraîne que la couche de sortie $(v, c)$.

In [ ]:
import math

---
# Partie A — Implémentation from scratch (Python pur)

### Question 1 — Lire la séquence 1, 2, 3 avec la cellule
**Réponse.** $h_1 = 0{,}5\times 0 + 1 = 1$ ; $h_2 = 0{,}5\times 1 + 2 = 2{,}5$ ; $h_3 = 0{,}5\times 2{,}5 + 3 = 4{,}25$.

In [ ]:
def lire_sequence(xs):
    # Cellule RNN simplifiée : h(t) = 0,5·h(t−1) + x(t)   (Wx = 1, Wh = 0,5, b = 0, sans tanh)
    h = 0.0
    etats = []
    for x in xs:
        h = 0.5 * h + x
        etats.append(h)
    return etats

sequence = [1, 2, 3]
etats = lire_sequence(sequence)
for t, e in enumerate(etats, start=1):
    if isinstance(e, tuple):
        print(f"t = {t} : x = {sequence[t - 1]} → c{t} = {e[0]:.4f} ; h{t} = {e[1]:.4f}")
    else:
        print(f"t = {t} : x = {sequence[t - 1]} → h{t} = {e:.4f}")
h_final = etats[-1][1] if isinstance(etats[-1], tuple) else etats[-1]
print("Mémoire finale transmise à la couche de sortie : h =", round(h_final, 4))

t = 1 : x = 1 → h1 = 1.0000
t = 2 : x = 2 → h2 = 2.5000
t = 3 : x = 3 → h3 = 4.2500
Mémoire finale transmise à la couche de sortie : h = 4.25


### Question 2 — Exemple 1 : entraîner la sortie de régression (diapo 84)
La couche de sortie est $\hat y = v\cdot h + c$, la perte $(y - \hat y)^2$ avec $y = 4$. Départ $v = c = 0$ ; $\eta = 0,02$ ; 3 epochs.

**Réponse.** Gradients : $\nabla v = -2(y - \hat y)\,h$ et $\nabla c = -2(y - \hat y)$. La mémoire $h$ ne change pas (la cellule est fixée) : seuls $v$ et $c$ bougent.

In [ ]:
def entrainer_sortie(h, y, eta, epochs=3, classification=False):
    v, b = 0.0, 0.0
    lignes = []
    for e in range(epochs + 1):
        z = v * h + b
        if classification:
            p = 1 / (1 + math.exp(-z))
            perte = -math.log(p) if y == 1 else -math.log(1 - p)
            delta = p - y
        else:
            p = z
            perte = (y - p) ** 2
            delta = -2 * (y - p)
        lignes.append((e, v, b, p, perte))
        v, b = v - eta * delta * h, b - eta * delta       # mise à jour (après la mesure)
    return lignes

reg = entrainer_sortie(h_final, 4, 0.02)
print("Epoch |    v       c    |   ŷ      perte")
for e, v, b, p, L in reg:
    print(f"  {e}   | {v:6.3f}  {b:6.3f} | {p:6.3f}  {L:7.3f}")

Epoch |    v       c    |   ŷ      perte
  0   |  0.000   0.000 |  0.000   16.000
  1   |  0.680   0.160 |  3.050    0.902
  2   |  0.842   0.198 |  3.774    0.051
  3   |  0.880   0.207 |  3.946    0.003


### Question 3 — Exemple 2 : entraîner la sortie de classification (diapo 85)
La série 1, 2, 3 est croissante ($y = 1$). Sortie $p = \sigma(v\cdot h + c)$, perte $-\ln p$, $\delta = p - y$ ; $\nabla v = \delta\,h$, $\nabla c = \delta$ ; $\eta = 0,1$.

**Réponse.** Même mémoire $h$, mais une **tête de classification** : la probabilité monte vers 1 et la perte baisse.

In [ ]:
cls = entrainer_sortie(h_final, 1, 0.1, classification=True)
print("Epoch |    v       c    |   p      perte")
for e, v, b, p, L in cls:
    print(f"  {e}   | {v:6.3f}  {b:6.3f} | {p:6.3f}  {L:7.3f}")

Epoch |    v       c    |   p      perte
  0   |  0.000   0.000 |  0.500    0.693
  1   |  0.213   0.050 |  0.722    0.326
  2   |  0.331   0.078 |  0.815    0.204
  3   |  0.409   0.096 |  0.862    0.148


---
# Partie B — Le même entraînement avec Keras

### Question 4 — Construire la cellule dans Keras
**Réponse.** Keras possède la couche `SimpleRNN`. Avec `activation=None` (pas de tanh), elle calcule exactement $h_t = W_x x_t + W_h h_{t-1} + b$. On impose $W_x = 1$, $W_h = 0{,}5$, $b = 0$ et on la **gèle** (`trainable=False`) : seule la couche `Dense` de sortie est entraînée, comme dans la diapo.

In [ ]:
import numpy as np
import keras
from keras import layers

def creer_cellule():
    cellule = layers.SimpleRNN(1, activation=None, trainable=False)
    return cellule

def regler_cellule(cellule):
    # poids : [W_x (entrée), W_h (mémoire), b]
    cellule.set_weights([np.array([[1.0]]), np.array([[0.5]]), np.array([0.0])])

class HistoriqueSortie(keras.callbacks.Callback):
    # Enregistre (v, b) de la couche Dense au début puis à la fin de chaque epoch
    def on_train_begin(self, logs=None):
        self.vb = [self._vb()]
    def on_epoch_end(self, epoch, logs=None):
        self.vb.append(self._vb())
    def _vb(self):
        k, c = self.model.layers[-1].get_weights()
        return float(k[0, 0]), float(c[0])

def entrainer_keras(eta, classification):
    cellule = creer_cellule()
    model = keras.Sequential([keras.Input(shape=(3, 1)), cellule,
                              layers.Dense(1, activation="sigmoid" if classification else None,
                                           kernel_initializer="zeros", bias_initializer="zeros")])
    regler_cellule(cellule)
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=eta),
                  loss="binary_crossentropy" if classification else "mse")
    X = np.array([[[1.0], [2.0], [3.0]]])                 # 1 séquence de 3 pas, 1 valeur par pas
    y = np.array([[1.0 if classification else 4.0]])
    hist = HistoriqueSortie()
    model.fit(X, y, epochs=3, batch_size=1, shuffle=False, verbose=0, callbacks=[hist])
    h_keras = float(keras.ops.convert_to_numpy(cellule(X))[0, 0])
    return model, hist.vb, h_keras

### Question 5 — Entraîner les deux têtes avec Keras
**Réponse.** Le modèle complet est `Input(3, 1)` → cellule (gelée) → `Dense(1)`. On utilise l’optimiseur `SGD` avec le même η et la même perte qu’en Python pur : `mse` pour la régression, `binary_crossentropy` pour la classification.

In [ ]:
m_reg, vb_reg, h_keras = entrainer_keras(0.02, classification=False)
m_cls, vb_cls, _ = entrainer_keras(0.1, classification=True)
X = np.array([[[1.0], [2.0], [3.0]]])
print("Mémoire h calculée par Keras :", round(h_keras, 4))
print("Régression     : (v, c) par epoch =", [tuple(round(x, 3) for x in t) for t in vb_reg],
      "| ŷ final =", round(float(m_reg.predict(X, verbose=0)[0, 0]), 3))
print("Classification : (v, c) par epoch =", [tuple(round(x, 3) for x in t) for t in vb_cls],
      "| p final =", round(float(m_cls.predict(X, verbose=0)[0, 0]), 3))

Mémoire h calculée par Keras : 4.25
Régression     : (v, c) par epoch = [(0.0, 0.0), (0.68, 0.16), (0.842, 0.198), (0.88, 0.207)] | ŷ final = 3.946
Classification : (v, c) par epoch = [(0.0, 0.0), (0.213, 0.05), (0.331, 0.078), (0.409, 0.096)] | p final = 0.862


---
# Partie C — Comparaison et validation
### Question 6 — Python pur et Keras donnent-ils les mêmes paramètres à chaque epoch ?

In [ ]:
import pandas as pd
lignes = []
for (e, v, b, p, L), (vk, bk) in zip(reg, vb_reg):
    lignes.append(("régression", e, round(v, 4), round(vk, 4), round(b, 4), round(bk, 4)))
for (e, v, b, p, L), (vk, bk) in zip(cls, vb_cls):
    lignes.append(("classification", e, round(v, 4), round(vk, 4), round(b, 4), round(bk, 4)))
print(f"Mémoire h : Python pur = {h_final:.4f} ; Keras = {h_keras:.4f}")
display(pd.DataFrame(lignes, columns=["Exemple", "Epoch", "v Python pur", "v Keras", "c Python pur", "c Keras"]))
assert abs(h_keras - h_final) < 1e-4
assert np.allclose([(v, b) for _, v, b, _, _ in reg], vb_reg, atol=1e-4)
assert np.allclose([(v, b) for _, v, b, _, _ in cls], vb_cls, atol=1e-4)
print("======= Python pur = Keras : même mémoire h et mêmes paramètres de sortie à chaque epoch")

Mémoire h : Python pur = 4.2500 ; Keras = 4.2500


,Exemple,Epoch,v Python pur,v Keras,c Python pur,c Keras
0,régression,0,0.0000,0.0000,0.0000,0.0000
1,régression,1,0.6800,0.6800,0.1600,0.1600
2,régression,2,0.8415,0.8415,0.1980,0.1980
3,régression,3,0.8799,0.8799,0.2070,0.2070
4,classification,0,0.0000,0.0000,0.0000,0.0000
5,classification,1,0.2125,0.2125,0.0500,0.0500
6,classification,2,0.3308,0.3308,0.0778,0.0778
7,classification,3,0.4093,0.4093,0.0963,0.0963


✅ Python pur = Keras : même mémoire h et mêmes paramètres de sortie à chaque epoch


## Conclusion



**À retenir**

- Un RNN lit la séquence pas à pas avec les **mêmes poids** à chaque pas ; $h_t$ est sa mémoire.
- Le gradient remonte le temps en étant multiplié par $W_h$ à chaque pas : sur de longues séquences, il **s’évanouit** (ou explose). C’est ce que corrigent le LSTM et le GRU.
- La même mémoire peut alimenter une sortie de régression (linéaire + MSE) ou de classification (sigmoïde + log-loss).